# mboostrs

**Component-wise functional gradient boosting for Python and R, with a Rust core.**

[Download this book as PDF](/exports/mboostrs.pdf) · [Download the notebooks](/exports/notebooks.zip)

`mboostrs` fits component-wise boosting models with a Rust engine and
Python and R interfaces. Its formula syntax and statistical methods follow
R's [`mboost`](https://cran.r-project.org/package=mboost) and
[`gamboostLSS`](https://cran.r-project.org/package=gamboostLSS).

The book is written for readers who know regression and want to fit, tune and
interpret boosting models; no prior knowledge of boosting is assumed. Read
[Installation](usage/install.ipynb), then the [Quickstart](usage/quickstart.ipynb).
Readers who want the derivations go to Part II.

The [capability reference](appendix/capabilities.ipynb) lists supported
operations, comparisons with R, test tolerances, and known differences.
Upstream's `bss` and `blackboost` learners and its `FP` and `RCG` families
are not implemented. Benchmark pages specify which fitting and prediction
steps are included in each timing.


::::{grid} 1 1 3 3

:::{grid-item-card} Part I — Using mboostrs
Task-oriented tutorials for fitting, selecting, diagnosing, and comparing
models. The build executes every code cell against the installed package.
:::

:::{grid-item-card} Part II — Methodology
Derivations and statistical explanations of component-wise functional gradient
boosting, penalized base learners, loss functions, distributional (GAMLSS)
boosting, multinomial models, and model selection.
:::

:::{grid-item-card} Part III — Case studies
Worked analyses of effect shapes, response distributions, validation, and
stopping-iteration selection on real and simulated datasets.
:::

:::{grid-item-card} Part IV — Implementation
The implementation of the boosting engine, linear-algebra kernels, penalized
least squares, multinomial Kronecker products, Python and R bindings, numerical
parity decisions, testing, and performance.
:::

:::{grid-item-card} Part V — Benchmarks
Comparisons scored against a known answer or another library: public data
sets with fixed seeds and data digests, run time and memory as data grow, and
agent analysts given the same brief with each library and graded on hidden
replicates.
:::

::::

## Worked analyses and comparisons

- [Munich rents: distributional modelling](cases/munich_rents_distributional.ipynb) follows a rent distribution from specification and grouped validation to interpreted predictions and a saved model.
- [Distributional regression: comparison across libraries](cases/distributional_comparison.ipynb) compares predictive distributions and function recovery in mboostrs, gamboostLSS, mssm and NGBoost.
- [Specification search: repeated validation](cases/specification_search_validation.ipynb) measures how the existing search changes prediction error and model size across independent training samples.

## Component-wise boosting

Statistical learning starts from a **loss function** $\rho(y, f)$ that scores
how badly a prediction $f$ misses the observed $y$ (squared error for
regression, log-loss for classification, …). The **empirical risk** is the
(weighted) sum of the loss over the $n$ training points,

$$
\mathcal{R}(f) = \sum_{i=1}^{n} w_i\, \rho\!\left(y_i, f(x_i)\right),
$$

with unit weights $w_i=1$ for unweighted data (this is the convention used
throughout the book). Learning means finding a predictor $f$ that makes $\mathcal{R}$ small. This is **empirical risk minimization (ERM)**, the principle behind
maximum-likelihood fitting, ordinary least squares, and neural-network training
alike. Component-wise boosting is a particular *algorithm* for ERM: instead of
optimizing all of $f$ at once, it lowers the risk in small steps, each of which
changes the contribution of one base learner.

Given a differentiable loss $\rho(y, f)$ and a set of simple **base learners**
(linear terms, P-splines, trees, spatial and random effects, …), component-wise
boosting builds an additive predictor

$$
\hat f(x) = \hat f^{[0]} + \nu \sum_{m=1}^{m_{\text{stop}}} \hat g^{[m]}(x),
$$

where $\hat f^{[0]}$ is the starting value (the offset), $\hat g^{[m]}$ is the
fit of the base learner selected at iteration $m$, $\nu$ is the learning rate
and $m_{\text{stop}}$ is the number of iterations.

At each iteration the procedure identifies the direction of steepest loss
decrease: the **negative gradient of $\rho$ with respect to $f$**, evaluated
at every observation. For squared loss this gradient is exactly the residual
$y - \hat f$; for other losses it is a generalized residual, often called the
**pseudo-residual**. The procedure then:

1. computes the pseudo-residual $u_i = -\partial \rho(y_i, f(x_i))/\partial f$
   at the current fit,
2. fits **every** base learner to that pseudo-residual by (penalized) least
   squares,
3. selects the learner with the smallest fitting criterion (this is the
   "component-wise" step), and
4. adds a small step $\nu$ (the **learning rate**, typically $0.1$) of it to
   the predictor.

The step length $\nu$ controls the size of each update, and the stopping
iteration $m_{\text{stop}}$ limits the cumulative path. In routine use $\nu$ is
fixed at a small value and $m_{\text{stop}}$ is selected by cross-validated risk.

An unpenalized least-squares fit estimates all coefficients jointly and is not
identified when the design has more columns than its rank. Boosting instead solves
one low-dimensional learner problem at a time. A selected learner may be updated
again later, but the other learners are unchanged in that iteration. Learners not
selected before stopping contribute zero, permitting sparse additive fits when
$p\gg n$ without inverting a full $p\times p$ system.

The same machinery applies to distributional regression: instead of one
predictor for the mean, each parameter of the response distribution (location,
scale, shape) gets its own additive predictor. This is the GAMLSS/`gamboostLSS`
setting.

## Capability summary

The table lists features available through Python and R. The generated
[capability reference](appendix/capabilities.ipynb) gives operation-level
support, validation tests, and recorded tolerances.

```{list-table}
:header-rows: 1

* - Capability
  - What is available
* - `glmboost`, `gamboost`, `cvrisk`, stability selection, inference
  - Python and R. Cross-validation and stability selection accept weights, so
    they work with every family that defines a weighted gradient.
* - Standard families
  - All 22: Gaussian, GaussReg, Binomial, Poisson, GammaReg, NBinomial, Hurdle, Huber,
    Laplace, Quantile, Expectile, AdaExp, HingeLoss, PropOdds, CoxPH, Weibull,
    Loglog, Lognormal, Multinomial, and the three ranking families in the row
    below. `GammaReg` is the Gamma regression family; there is no `Gamma`,
    because in R that name would silently resolve to `stats::Gamma`.
    All fit and predict on the link scale; response-scale prediction, AIC, and
    diagnostics vary by family — see [](usage/interface_map).
* - Ranking and survival-rank families
  - `AUC`, `Gehan`, and `Cindex` in Python and R, with weighted gradients, so
    they also work under `cvrisk` and stability selection.
* - Base learners
  - `bols`, `bbs`, `bns`, `bmono`, `bconstrained`, `bmrf`, `bspatial`,
    `brandom`, `btree`, `bkernel`, `buser` in Python and R. All eleven fit under
    `gamboost` and `gamboostLSS`; `glmboostLSS` fits linear terms only, so it
    takes `bols` (or a bare column name) and rejects the rest. `blackboost`,
    `brad`, and `bss` are not implemented.
* - `gamboostLSS` / `glmboostLSS`
  - Python and R: fitting, cyclic and non-cyclic paths, cross-validation, and
    stability selection, across Gaussian, Gamma, StudentT, Beta, NBinomial,
    LogNormal, LogLog, Weibull, ZIPo, ZINB, Dirichlet and more. Distributional
    diagnostics (PIT, quantile residuals, centiles) need a family CDF or
    quantile function. Gaussian, NBinomial, ZIPo, Poisson, Geometric, Binomial,
    Gamma, StudentT, Beta, and ZINB supply both; the remaining families fit and
    predict but do not carry the full diagnostic chain.
* - Multinomial smooth effects
  - Python and R, via the Kronecker path, including new-data and
    truncated-`mstop` prediction. Residuals, AIC, and hat values are unavailable
    for this family.
* - Prediction intervals
  - Split-conformal, CV+, and conformalized quantile regression for any standard
    model, in Python. Distributional models get intervals from the fitted
    distribution instead.
* - Formula conveniences
  - Whole-frame `.` expansion, term removal, inline transforms such as `log(x)`
    and `scale(x)`, automatic categorical expansion, `na_action="omit"`, and
    `%X%` interaction learners. Python. Intercept control (`- 1`, `+ 0`) is not
    supported; see [](usage/r_migration) for the equivalent.
* - Portable model artifacts
  - Schema-versioned JSON written by Python `save_model`, and read back by
    Python, R, and the `mboostrs-core` Rust crate, so a fitted model can be
    scored without Python. Linear models score in all three; spline and
    transformed models load for inspection everywhere and score in Python.
* - Stochastic boosting
  - Base-learner (component) subsampling in the engine. Exposed as `subsample=`
    on `gamboost` in Python and on `boost_control` in R. Row subsampling is not
    implemented.
* - Warm continuation
  - `continue_to(m)` returns the model grown to a larger `mstop`, equal to a
  cold fit at `mstop=m`. A gamboost fit whose family kept the gradient it
  stopped on resumes from there; the Gaussian Gram route, `glmboost` and
  the distributional models refit from iteration zero and save only the
  re-specification.
* - scikit-learn interface
  - Python. Regressors and classifiers including multiclass, quantile,
    tree-based, distributional, and survival estimators, with `sample_weight`
    support and tags that state their real limits.
* - Plotting and diagnostics
  - Python. Risk paths, selection frequencies, partial effects, residual and
    calibration frames, PIT and worm-plot data, and Q-statistics.
```

```{tip}
If you are migrating from R, start with [](usage/r_migration) and the runtime
capability tables in [](usage/interface_map). Part IV documents the places where
this package intentionally differs from R, with the reasoning.
```

## Bibliographic Notes

Tukey's **twicing** is an early residual-fitting procedure
{cite}`tukey1977exploratory`: fit a smoother $S$ to the data, fit the *same*
smoother to the residuals, and add the second fit to the first, giving
$S(y) + S(y - S(y))$. That is one boosting iteration with step length
$\nu = 1$. Repeating residual updates with a small step length and selecting
from a set of candidate smoothers gives component-wise boosting.

In machine learning, boosting started from a different question: whether a learner only
slightly better than chance can be combined into an arbitrarily accurate one.
Kearns and Valiant posed it, Schapire answered it constructively, and AdaBoost
made it practical {cite}`kearns1989cryptographic,schapire1990strength,freund1997decision`.
Friedman et al. showed that AdaBoost is stagewise fitting of an additive
model under exponential loss {cite}`friedman2000additive`; Mason et al. then
interpreted boosting as gradient descent in function space
{cite}`mason2000boosting`. This package uses forward stagewise functional
gradient descent, regularized by shrinkage and early stopping
{cite}`buhlmann2007boosting`.
[](methodology/from_glm_to_boosting.ipynb) develops the connection in detail.

The component-wise algorithm implemented here is due to
{cite}`buhlmann2003boosting`, {cite}`buhlmann2007boosting`, and
{cite}`hothorn2010mboost`. The other components come from
{cite}`eilers1996flexible` (P-splines), {cite}`rigby2005generalized` and
{cite}`mayr2012generalized` (GAMLSS boosting), {cite}`thomas2018gradient`
(non-cyclic updates), and {cite}`meinshausen2010stability` with
{cite}`hofner2015controlling` (stability selection).

For hands-on tutorials, see {cite}`hofner2014mboost_tutorial` for `mboost` and
{cite}`hofner2014gamboostlss` for `gamboostLSS`. {cite}`mayr2014extending` is a
broad overview of statistical boosting.

The next page to read is [Installation](usage/install.ipynb).

```{bibliography}
:filter: docname in docnames
```
